In [ ]:
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels, user_study_tasks_only, user_study_nice_model_names, y_cls_to_nice_name, user_study_plot_hist, set_seed
from nocode_robot_programming.state_decision.AEGP_model import AEGP
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda
from nocode_robot_programming.state_decision_dataset_prepare.dataset_auto import TrajectoryDatasetEvaluationViewBuilder
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
kill_other_ipykernels(force=True)
set_seed()

dataset_builder = TrajectoryDatasetEvaluationViewBuilder()
datasets = dataset_builder.load_eval_from_task("petr_kin_peg_pick")

# Video embedding bench

- I took the videoembedding from ILeSiA project, this model should be able to reconstruct images with details
#### 1. The autoencoder model very blurry reconstructions when training on a small dataset (~25 images), see here:

In [ ]:
aegp = AEGP(pix=64)
d_train, d_test, _ = datasets[0]

aegp.y_cls = d_train.y_cls
X = aegp._dataset_prepare(d_train.X)
aegp.videoembedder.training_loop(DataLoader(X, batch_size=1), num_epochs=50)

plt.plot(np.array([aegp.videoembedder.losses1, aegp.videoembedder.losses2]).T)
ipt.save(); ipt.show()
reconstructed_images = aegp.videoembedder.model.forward(aegp.tf(d_train.X).unsqueeze(1)).squeeze()
display(show_gray_video_cuda(torch.concatenate([aegp.tf(d_train.X), reconstructed_images], dim=2), fps=20, scale=5))

Problems: BatchNorm and tiny batches + small data

----

### 2. Repeat the input data 10x:

In [ ]:
from copy import deepcopy
def dupl(dataset, n=5):
    dataset_dupl = deepcopy(dataset)
    dataset_dupl.X = torch.vstack([dataset.X] * n)
    dataset_dupl.y_int = torch.tile(dataset.y_int, dims=(n,))
    dataset_dupl.y_names = [*dataset.y_names] * n
    return dataset_dupl

In [ ]:
for d_train, d_test, _ in datasets:
    for i in [1,10,20]:
        print("------------------------------------------------------")
        print(f"Samples: {d_train.X.size(0)}, dupl: {i}")
        aegp = AEGP(pix=224)
        d_train_dupl = dupl(d_train, n=i)

        aegp.y_cls = d_train.y_cls
        X = aegp._dataset_prepare(d_train_dupl.X)
        aegp.videoembedder.training_loop(DataLoader(X, batch_size=1), num_epochs=50)

        plt.plot(np.array([aegp.videoembedder.losses1, aegp.videoembedder.losses2]).T)
        ipt.save(); ipt.show()
        reconstructed_images = aegp.videoembedder.model.forward(aegp.tf(d_train.X).unsqueeze(1)).squeeze()
        display(show_gray_video_cuda(torch.concatenate([aegp.tf(d_train.X), reconstructed_images], dim=2), fps=20, scale=5))

# Conclusion
### Configuration
- `batch_size=1`, `shuffle=False`, `drop_last=False`
- `3 x 3` runs - (trials split into train/test sets x duplication 1,10,20)
### Visual Evaluation
- 2 train trials + no duplication -> 1/10
- 2 train trials + 10 duplication -> 2/10
- 2 train trials + 20 duplication -> 3/10 (flickering)
- 5 train trials + no duplication -> 3/10 
- 5 train trials + 10 duplication -> 4/10 
- 5 train trials + 20 duplication -> 4.5/10 (heavy flickering)
- 8 train trials + no duplication -> 4
- 8 train trials + 10 duplication -> 5 (peg not reconstructed)
- 8 train trials + 20 duplication -> 6/10 (peg not reconstructed sometimes + flickering)

### Configuration
- `batch_size=1`, `shuffle=True`, `drop_last=True`
- `3 x 3` runs - (trials split into train/test sets x duplication 1,10,20)
### Visual Evaluation
- 2 train trials + no duplication -> 1.5/10
- 2 train trials + 10 duplication -> 5/10 (slight flickering)
- 2 train trials + 20 duplication -> 6/10 (slight flickering)
- 5 train trials + no duplication -> 2/10 
- 5 train trials + 10 duplication -> 3/10 (a lot of flickering, this should be much better but is not) 
- 5 train trials + 20 duplication -> 4/10 (a lot of flickering)
- 8 train trials + no duplication -> 3/10
- 8 train trials + 10 duplication -> 3.5/10 (peg not reconstructed)
- 8 train trials + 20 duplication -> 4.5/10 (peg sometimes not reconstructed)

### Configuration
- `batch_size=32`, `shuffle=True`, `drop_last=True`
- `3 x 3` runs - (trials split into train/test sets x duplication 1,10,20)
### Visual Evaluation
- 2 train trials + no duplication -> 0/10
- 2 train trials + 10 duplication -> 1/10 
- 2 train trials + 20 duplication -> 2/10 
- 5 train trials + no duplication -> 2/10 
- 5 train trials + 10 duplication -> 2/10
- 5 train trials + 20 duplication -> 4/10 (can see some peg, but flickering a lot) 
- 8 train trials + no duplication -> 1/10
- 8 train trials + 10 duplication -> 2.5/10 
- 8 train trials + 20 duplication -> 5/10 (peg kind of reconstructed, slight flickering)


In [ ]:
for d_train, d_test, _ in datasets:
    for i in [1,10,20]:
        print("------------------------------------------------------")
        print(f"Samples: {d_train.X.size(0)}, dupl: {i}")
        aegp = AEGP(pix=224)
        d_train_dupl = dupl(d_train, n=i)

        aegp.y_cls = d_train.y_cls
        X = aegp._dataset_prepare(d_train_dupl.X)
        aegp.videoembedder.training_loop(DataLoader(X, batch_size=1, shuffle=True, drop_last=True), num_epochs=50)

        plt.plot(np.array([aegp.videoembedder.losses1, aegp.videoembedder.losses2]).T)
        ipt.save(); ipt.show()
        reconstructed_images = aegp.videoembedder.model.forward(d_train.X.unsqueeze(1)).squeeze()
        display(show_gray_video_cuda(torch.concatenate([d_train.X, reconstructed_images], dim=2), fps=20, scale=5))

In [ ]:
for d_train, d_test, _ in datasets:
    for i in [1,10,20]:
        print("------------------------------------------------------")
        print(f"Samples: {d_train.X.size(0)}, dupl: {i}")
        aegp = AEGP(pix=224)
        d_train_dupl = dupl(d_train, n=i)

        aegp.y_cls = d_train.y_cls
        X = aegp._dataset_prepare(d_train_dupl.X)
        aegp.videoembedder.training_loop(DataLoader(X, batch_size=32, shuffle=True, drop_last=True), num_epochs=50)

        plt.plot(np.array([aegp.videoembedder.losses1, aegp.videoembedder.losses2]).T)
        ipt.save(); ipt.show()
        reconstructed_images = aegp.videoembedder.model.forward(d_train.X.unsqueeze(1)).squeeze()
        display(show_gray_video_cuda(torch.concatenate([d_train.X, reconstructed_images], dim=2), fps=20, scale=5))

# Winner?

In [ ]:
d_train, d_test, _ = datasets[0]
print(f"Samples: {d_train.X.size(0)}, dupl: 20")
aegp = AEGP(pix=224)
d_train_dupl = dupl(d_train, n=20)

aegp.y_cls = d_train.y_cls
X = aegp._dataset_prepare(d_train_dupl.X)
aegp.videoembedder.training_loop(DataLoader(X, batch_size=1, shuffle=True, drop_last=True), num_epochs=50)

plt.plot(np.array([aegp.videoembedder.losses1, aegp.videoembedder.losses2]).T)
ipt.save(); ipt.show()
reconstructed_images = aegp.videoembedder.model.forward(d_train.X.unsqueeze(1)).squeeze()
display(show_gray_video_cuda(torch.concatenate([d_train.X, reconstructed_images], dim=2), fps=20, scale=5))

In [ ]:
d_train_dupl.X.shape

In [ ]:
import torch
import torchvision.transforms.functional as F
from torch.utils.data import Dataset

class TransformDataset(Dataset):
    def __init__(self, X, size=64, p_hflip=0.5):
        self.X = X                  # expect shape (N, 2, H, W) or list of two PILs per item
        self.size = size
        self.p_hflip = p_hflip

    def _paired_aug(self, x0, x1):
        # x*: tensor (1,H,W) or (H,W); values in [0,1]
        if x0.ndim == 2:  # make (1,H,W)
            x0 = x0.unsqueeze(0); x1 = x1.unsqueeze(0)

        H, W = x0.shape[-2:]
        # sample once
        do_hflip = torch.rand(()) < self.p_hflip
        angle = (torch.rand(()) * 10.0) - 5.0  # [-5°, +5°]
        do_crop = torch.rand(()) < 0.5
        if do_crop:
            scale = torch.empty(()).uniform_(0.90, 1.00)
            ratio = torch.empty(()).uniform_(0.98, 1.02)
            target = (H*W) * scale
            new_w = torch.clamp(torch.sqrt(target * ratio).round().long(), 1, W)
            new_h = torch.clamp(torch.sqrt(target / ratio).round().long(), 1, H)
            i = int(torch.randint(int((H-new_h).clamp(min=0)+1), ())) if new_h < H else 0
            j = int(torch.randint(int((W-new_w).clamp(min=0)+1), ())) if new_w < W else 0
            def crop_resize(z): return F.resized_crop(z, i, j, int(new_h), int(new_w), [self.size, self.size])
        else:
            def crop_resize(z): return F.resize(z, [self.size, self.size], antialias=True) if (H,W)!=(self.size,self.size) else z

        def apply(z):
            if do_hflip: z = F.hflip(z)
            z = crop_resize(z)
            if angle.abs() > 0:
                z = F.rotate(z, float(angle), interpolation=F.InterpolationMode.BILINEAR, fill=0.0)
            return z.clamp(0, 1)

        return apply(x0), apply(x1)

    def __len__(self): return len(self.X)

    def __getitem__(self, idx):
        # assume tensors in [0,1]; if you store PIL images, call F.to_tensor first
        x0 = self.X[idx, 0]
        x1 = self.X[idx, 1]
        x0, x1 = self._paired_aug(x0, x1)
        return torch.cat([x0, x1], dim=0)   # (2, 64, 64)

In [ ]:
TransformDataset(X)[0]

In [ ]:
X.shape

In [ ]:
X = aegp._dataset_prepare(d_train_dupl.X)

d_train_X_tf_list = []
for i in range(10):
    d_train_X_tf_list.append(TransformDataset(X)[np.random.randint(420)])
d_train_X_tf = torch.concatenate(d_train_X_tf_list, dim=1)
display(show_gray_video_cuda(d_train_X_tf, fps=20, scale=5))

In [ ]:
d_train_X_tf.shape

In [ ]:
aegp = AEGP(pix=224)

aegp.y_cls = d_train.y_cls
X = aegp._dataset_prepare(d_train_X_tf)
aegp.videoembedder.training_loop(DataLoader(TransformDataset(X), batch_size=1, shuffle=True, drop_last=True), num_epochs=100)

plt.plot(np.array([aegp.videoembedder.losses1, aegp.videoembedder.losses2]).T)
ipt.save(); ipt.show()
reconstructed_images = aegp.videoembedder.model.forward(d_train.X.unsqueeze(1)).squeeze()
display(show_gray_video_cuda(torch.concatenate([d_train.X, reconstructed_images], dim=2), fps=20, scale=5))

- Reconstruction is worse, when added augmentation